# NPS Driver Analysis

## Project Overview
Analyze Net Promoter Score (NPS) survey results and supporting customer features to understand what differentiates promoters (9-10), passives (7-8), and detractors (0-6). Identify the key drivers that push customers toward or away from recommending the product.

## Learning Objectives
- Understand NPS methodology and its business implications
- Segment survey respondents into promoter/passive/detractor groups
- Identify features and behaviors that discriminate between NPS segments
- Derive actionable recommendations to improve NPS

## Problem Statement
A company's overall NPS is stagnant. Leadership wants to understand what specific aspects of the product and customer experience differentiate promoters from detractors so they can prioritize improvement efforts.

## Why This Project Matters
NPS is one of the most widely tracked customer experience metrics. Understanding its drivers enables targeted improvements: fixing what detractors hate and amplifying what promoters love. A 10-point NPS improvement often correlates with measurable revenue growth.

## Dataset Overview
Synthetic NPS survey data: ~2K respondents with NPS score, satisfaction sub-scores (product, support, price, onboarding), usage metrics, plan tier, and tenure.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- Patterns inspired by typical SaaS NPS surveys
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_respondents = 2000

plans = ['Free', 'Pro', 'Enterprise']
plan_weights = [0.40, 0.40, 0.20]
tenure_months = np.random.exponential(scale=12, size=n_respondents).astype(int) + 1
tenure_months = np.clip(tenure_months, 1, 60)

rows = []
for i in range(n_respondents):
    plan = np.random.choice(plans, p=plan_weights)
    tenure = tenure_months[i]

    # Sub-scores (1-5 scale) drive NPS
    product_sat = np.clip(np.random.normal(3.5 if plan != 'Free' else 3.0, 0.8), 1, 5)
    support_sat = np.clip(np.random.normal(3.2, 1.0), 1, 5)
    price_sat = np.clip(np.random.normal(3.0 if plan == 'Enterprise' else 3.5 if plan == 'Free' else 3.3, 0.9), 1, 5)
    onboarding_sat = np.clip(np.random.normal(3.4 if tenure > 6 else 2.8, 0.8), 1, 5)

    # Usage features
    login_freq = max(1, int(np.random.exponential(scale=10 if plan != 'Free' else 5)))
    features_used = np.random.randint(1, 9 if plan == 'Enterprise' else 6 if plan == 'Pro' else 4)
    support_tickets = np.random.poisson(2 if support_sat < 3 else 0.5)

    # NPS driven by sub-scores + noise
    nps_driver = (product_sat * 0.35 + support_sat * 0.20 + price_sat * 0.20 + onboarding_sat * 0.25)
    nps_score = int(np.clip(np.round(nps_driver * 2 + np.random.normal(0, 0.8)), 0, 10))

    rows.append({
        'RespondentID': f'R{i:05d}', 'Plan': plan, 'TenureMonths': tenure,
        'ProductSatisfaction': round(product_sat, 1),
        'SupportSatisfaction': round(support_sat, 1),
        'PriceSatisfaction': round(price_sat, 1),
        'OnboardingSatisfaction': round(onboarding_sat, 1),
        'LoginFrequency': login_freq, 'FeaturesUsed': features_used,
        'SupportTickets': support_tickets, 'NPSScore': nps_score,
    })

df = pd.DataFrame(rows)

# NPS category
def nps_category(score):
    if score >= 9: return 'Promoter'
    elif score >= 7: return 'Passive'
    else: return 'Detractor'

df['NPSCategory'] = df['NPSScore'].apply(nps_category)
print(f'Dataset shape: {df.shape}')
print(f'NPS distribution:\n{df["NPSCategory"].value_counts()}')

# Calculate NPS
promoters_pct = (df['NPSCategory'] == 'Promoter').mean() * 100
detractors_pct = (df['NPSCategory'] == 'Detractor').mean() * 100
nps = promoters_pct - detractors_pct
print(f'\nOverall NPS: {nps:.0f}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'NPS Score range: {df["NPSScore"].min()} - {df["NPSScore"].max()}')
print(f'Satisfaction ranges:')
for col in ['ProductSatisfaction', 'SupportSatisfaction', 'PriceSatisfaction', 'OnboardingSatisfaction']:
    print(f'  {col}: {df[col].min():.1f} - {df[col].max():.1f}')
print(f'Tenure range: {df["TenureMonths"].min()} - {df["TenureMonths"].max()} months')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# NPS score distribution - colored by category
score_counts = df['NPSScore'].value_counts().sort_index()
bar_colors = ['#ef5350' if s <= 6 else '#ffa726' if s <= 8 else '#66bb6a' for s in score_counts.index]
axes[0,0].bar(score_counts.index, score_counts.values, color=bar_colors, edgecolor='black', alpha=0.7)
axes[0,0].set_title('NPS Score Distribution')
axes[0,0].set_xlabel('NPS Score')

# NPS category breakdown
colors = {'Promoter': '#66bb6a', 'Passive': '#ffa726', 'Detractor': '#ef5350'}
df['NPSCategory'].value_counts().reindex(['Promoter','Passive','Detractor']).plot.bar(
    ax=axes[0,1], color=[colors[c] for c in ['Promoter','Passive','Detractor']], edgecolor='black')
axes[0,1].set_title('NPS Category Distribution')
axes[0,1].tick_params(axis='x', rotation=0)

# NPS by plan
plan_nps = df.groupby('Plan').apply(
    lambda x: (x['NPSCategory'] == 'Promoter').mean()*100 - (x['NPSCategory'] == 'Detractor').mean()*100
).reindex(['Free','Pro','Enterprise'])
plan_nps.plot.bar(ax=axes[1,0], edgecolor='black', color='steelblue')
axes[1,0].set_title('NPS by Plan Tier')
axes[1,0].set_ylabel('NPS')
axes[1,0].axhline(y=0, color='red', linestyle='--')
axes[1,0].tick_params(axis='x', rotation=0)

# Satisfaction sub-scores by category
sat_cols = ['ProductSatisfaction', 'SupportSatisfaction', 'PriceSatisfaction', 'OnboardingSatisfaction']
cat_means = df.groupby('NPSCategory')[sat_cols].mean().reindex(['Promoter','Passive','Detractor'])
cat_means.plot.bar(ax=axes[1,1], edgecolor='black')
axes[1,1].set_title('Avg Satisfaction Scores by NPS Category')
axes[1,1].set_ylabel('Score (1-5)')
axes[1,1].tick_params(axis='x', rotation=0)
axes[1,1].legend(title='Dimension', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Target Analysis — NPS Drivers

In [ ]:
# Correlation between features and NPS score
driver_cols = sat_cols + ['LoginFrequency', 'FeaturesUsed', 'SupportTickets', 'TenureMonths']
corr_with_nps = df[driver_cols + ['NPSScore']].corr()['NPSScore'].drop('NPSScore').sort_values(ascending=False)
print('Correlation with NPS Score:')
print(corr_with_nps.round(3))

fig, ax = plt.subplots(figsize=(10, 5))
corr_with_nps.plot.barh(ax=ax, edgecolor='black', color=['green' if v > 0 else 'red' for v in corr_with_nps])
ax.set_title('Feature Correlation with NPS Score')
ax.axvline(x=0, color='grey', linestyle='--')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'nps_drivers.png'), dpi=100, bbox_inches='tight')
plt.show()

## Promoter vs Detractor Profile Comparison

In [ ]:
profile = df.groupby('NPSCategory')[driver_cols].mean().reindex(['Promoter','Passive','Detractor']).T
profile['Promoter_vs_Detractor_Gap'] = profile['Promoter'] - profile['Detractor']
profile = profile.sort_values('Promoter_vs_Detractor_Gap', ascending=False)
print('Profile Comparison (Promoter vs Detractor gap):')
print(profile.round(2))

## Satisfaction Gap Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
gap = profile['Promoter_vs_Detractor_Gap'].sort_values()
gap.plot.barh(ax=ax, edgecolor='black', color=['#ef5350' if v < 0 else '#66bb6a' for v in gap])
ax.set_title('Promoter - Detractor Gap by Feature')
ax.set_xlabel('Gap (higher = promoters score much higher)')
ax.axvline(x=0, color='grey', linestyle='--')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'satisfaction_gap.png'), dpi=100, bbox_inches='tight')
plt.show()

## NPS by Tenure Cohort

In [ ]:
df['TenureBucket'] = pd.cut(df['TenureMonths'], bins=[0, 3, 6, 12, 24, 60],
                             labels=['0-3m', '3-6m', '6-12m', '12-24m', '24m+'])
tenure_nps = df.groupby('TenureBucket').apply(
    lambda x: (x['NPSCategory'] == 'Promoter').mean()*100 - (x['NPSCategory'] == 'Detractor').mean()*100
)
fig, ax = plt.subplots(figsize=(8, 5))
tenure_nps.plot.bar(ax=ax, edgecolor='black', color='mediumpurple')
ax.set_title('NPS by Customer Tenure')
ax.set_ylabel('NPS')
ax.set_xlabel('Tenure')
ax.axhline(y=0, color='red', linestyle='--')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'nps_by_tenure.png'), dpi=100, bbox_inches='tight')
plt.show()
print(tenure_nps.round(1))

## Key Satisfaction Dimensions — Boxplot Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
order = ['Detractor', 'Passive', 'Promoter']
palette = {'Detractor': '#ef5350', 'Passive': '#ffa726', 'Promoter': '#66bb6a'}
for ax, col in zip(axes.flat, sat_cols):
    sns.boxplot(data=df, x='NPSCategory', y=col, order=order, palette=palette, ax=ax)
    ax.set_title(col.replace('Satisfaction', ' Satisfaction'))
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'satisfaction_boxplots.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Product Satisfaction** is the #1 NPS driver — the largest gap between promoters and detractors
- **Onboarding Satisfaction** is the #2 driver, especially impactful for newer customers (0-6 month tenure)
- **Support Tickets** negatively correlate with NPS — customers who need frequent support are less satisfied
- **Enterprise** users have the highest NPS, likely due to dedicated support and deeper product usage
- **Price Satisfaction** shows a moderate gap — pricing alone doesn't make promoters, but poor perceived value creates detractors
- Early-tenure customers (0-3m) have the lowest NPS, pointing to onboarding friction

## Limitations
- Synthetic data with engineered correlations; real NPS drivers are harder to isolate
- Survey-based analysis — respondents may not represent all customers (non-response bias)
- Satisfaction sub-scores are self-reported and may not reflect actual experience
- No open-ended comment analysis, which often reveals the "why" behind scores

## How to Improve This Project
- Include open-ended NPS comments and apply text analysis to surface qualitative themes
- Use real survey data with higher-dimensional features
- Build a classification model to predict NPS category from behavioral/satisfaction features
- Add competitive benchmark comparison
- Track NPS over time to measure impact of product changes

## Production Considerations
- Run NPS surveys on a rolling basis (not just annually)
- Close the loop: follow up with detractors within 48 hours
- Track NPS by segment in executive dashboards
- Correlate NPS changes with product releases and support initiatives

## Common Mistakes
- Focusing only on the overall NPS number without segmenting
- Treating passives as "fine" — they're the most likely to switch to competitors
- Surveying too frequently and causing survey fatigue
- Not acting on results — NPS programs without follow-up action lose credibility

## Mini Challenge / Exercises
1. What would NPS be if all "Passive" respondents became "Promoters"?
2. Which plan tier has the highest detractor rate? What satisfaction dimension is weakest?
3. Create a simple scoring model: assign weights to each satisfaction dimension to predict NPS.
4. If you could improve one satisfaction dimension by 1 point for all customers, which would have the biggest NPS impact?

## Final Summary / Key Takeaways
- NPS is most useful when decomposed into its underlying drivers
- Product quality and onboarding experience are typically the strongest NPS levers
- Segmenting by plan tier and tenure reveals where to focus improvement efforts
- Support ticket volume is a leading indicator of detractor risk
- The goal is not just to measure NPS but to identify and act on the levers that move it